# Widget Example Using bqplot

See https://astrowidgets.readthedocs.io for additional details about the widget, including installation notes.

In [ ]:
%load_ext autoreload
%autoreload 2
from astropy.visualization import AsymmetricPercentileInterval
from astrowidgets.bqplot import ImageWidget

In [ ]:
from ginga.misc.log import get_logger

logger = get_logger('my viewer', log_stderr=True,
                    log_file=None, level=30)

In [ ]:
w = ImageWidget()

For this example, we use an image from Astropy data repository and load it as `CCDData`. Feel free to modify `filename` to point to your desired image.

Alternately, for local FITS file, you could load it like this instead:
```python
w.load_image(filename)
```    
Or if you wish to load a data array natively (without WCS):
```python
from astropy.io import fits
import numpy as np
with fits.open(filename, memmap=False) as pf:
    arr = pf[numhdu].data.copy()
w.load_image(arr)
```

In [ ]:
filename = 'http://data.astropy.org/photometry/spitzer_example_image.fits'
numhdu = 0

# Loads NDData
# NOTE: memmap=False is needed for remote data on Windows.
# NOTE: Some file also requires unit to be explicitly set in CCDData.
from astropy.nddata import CCDData
ccd = CCDData.read(filename, hdu=numhdu, format='fits')
w.load_image(ccd)

A viewer will be shown after running the next cell.
In Jupyter Lab, you can split it out into a separate view by right-clicking on the viewer and then select
"Create New View for Output". Then, you can drag the new
"Output View" tab, say, to the right side of the workspace. Both viewers are connected to the same events.

In [ ]:
w

## Manipulating the widget with the API

In [ ]:
# Use API methods for viewport control

w.set_viewport(center=(0, 0))
# Note: zoom_level is not part of the API, use field of view instead
vport = w.get_viewport()
w.set_viewport(fov=vport['fov'] / 5)  # Zoom in 5x by reducing FOV

In [ ]:
# Set a colormap
w.set_colormap('Greys_r')

In [ ]:
# Get current cuts using API
print("Current cuts:", w.get_cuts())

In [ ]:
# Set cuts using API -- any astropy interval works
from astropy.visualization import AsymmetricPercentileInterval
w.set_cuts(AsymmetricPercentileInterval(1, 99.9))

In [ ]:
# Get image data and access pixel values using API
img_data = w.get_image()
if hasattr(img_data, 'data'):
    print("Pixel value at (240, 120):", img_data.data[240, 120])
    print("Data shape:", img_data.data.shape)
else:
    print("Pixel value at (240, 120):", img_data[240, 120])
    print("Data shape:", img_data.shape)

In [ ]:
# Get cuts information using API
cuts = w.get_cuts()
if hasattr(cuts, 'upper_percentile'):
    print("Upper percentile:", cuts.upper_percentile)
else:
    print("Current cuts:", cuts)

The rest of the calls demonstrate how the widget API works. Comment/uncomment as needed. Feel free to experiment.

In [ ]:
# Programmatically center to (X, Y) on viewer using set_viewport
w.set_viewport(center=(1, 1))

In [ ]:
# Note: offset_by is not part of the astro-image-display-api
# Instead, get current center and offset from it
vport = w.get_viewport(sky_or_pixel='pixel')
current_center = vport['center']
if isinstance(current_center, tuple):
    # Pixel coordinates
    new_center = (current_center[0] + 100, current_center[1] + 100)
    w.set_viewport(center=new_center)

In [ ]:
from astropy.coordinates import SkyCoord

# Change the values here if you are not using given
# example image.
ra_str = '01h13m23.193s'
dec_str = '+00d12m32.19s'
frame = 'galactic'

# Programmatically center to SkyCoord on viewer using set_viewport
w.set_viewport(center=SkyCoord(ra_str, dec_str, frame=frame))

In [ ]:
from astropy import units as u
import numpy as np

# Change the values if needed.
deg_offset = 0.001 * u.deg

# Note: offset_by with SkyCoord is not part of the astro-image-display-api
# Instead, get current center and offset from it
vport = w.get_viewport()
current_center = vport['center']
if hasattr(current_center, 'ra'):
    # SkyCoord center - offset in RA/Dec
    new_ra = current_center.ra + deg_offset / np.cos(current_center.dec)
    new_dec = current_center.dec + deg_offset
    new_center = SkyCoord(new_ra, new_dec, frame=current_center.frame)
    w.set_viewport(center=new_center)

In [ ]:
current_center

In [ ]:
# Note: zoom_level is not part of the astro-image-display-api
# Instead, show current field of view
vport = w.get_viewport()
print("Current field of view:", vport['fov'])

In [ ]:
# Note: zoom() is not part of the astro-image-display-api
# Instead, set a specific field of view to zoom in/out
# Get current FOV and reduce it by half to zoom in 2x
vport = w.get_viewport()
current_fov = vport['fov']
if hasattr(current_fov, 'value'):
    # Angular FOV with units
    new_fov = current_fov / 2
    w.set_viewport(fov=new_fov)
else:
    # Pixel FOV  
    new_fov = current_fov / 2
    w.set_viewport(fov=new_fov)

In [ ]:
from astropy import units as u
w.set_viewport(fov=0.5 *u.degree)

In [ ]:
# Capture what viewer is showing and save RGB image.
w.save('test_bqplot.png', overwrite=True)

In [ ]:
# Note: stretch_options is not part of the astro-image-display-api
# The API uses astropy visualization stretch objects
from astropy.visualization import LinearStretch, LogStretch, SqrtStretch, PowerStretch
print("Available stretch types: LinearStretch, LogStretch, SqrtStretch, PowerStretch, etc.")

In [ ]:
# Get image stretch algorithm in use
print(w.get_stretch())

In [ ]:
# Change the stretch using astropy visualization objects
from astropy.visualization import LogStretch
w.set_stretch(LogStretch())
print(w.get_stretch())

In [ ]:
# Note: autocut_options is not part of the astro-image-display-api
# The API uses astropy visualization interval objects
from astropy.visualization import MinMaxInterval, PercentileInterval, AsymmetricPercentileInterval
print("Available cuts types: MinMaxInterval, PercentileInterval, AsymmetricPercentileInterval, etc.")

In [ ]:
# Get image cut levels in use
print(w.get_cuts())

In [ ]:
# Change the cuts by providing explicit low/high values
w.set_cuts((0, 100))
print(w.get_cuts())

In [ ]:
# Change the cuts with an astropy visualization interval object
from astropy.visualization import PercentileInterval
w.set_cuts(PercentileInterval(95))
print(w.get_cuts())

## Marker/catalog interaction using the API

### Catalog/marker management using the astro-image-display-api

In [ ]:
# Create a sample catalog to demonstrate catalog loading
import numpy as np
from astropy.table import Table

# Create some sample marker positions
n_markers = 5
x_coords = np.random.uniform(50, 200, n_markers)
y_coords = np.random.uniform(50, 150, n_markers)
sample_catalog = Table([x_coords, y_coords], names=['x', 'y'])

# Load catalog using the API
w.load_catalog(sample_catalog, catalog_label='sample')
print("Loaded catalog with", len(sample_catalog), "entries")

In [ ]:
# Get catalog data using the API
retrieved_catalog = w.get_catalog(catalog_label='sample')

# Display the catalog nicely
print('{:^8s} {:^8s}'.format('X', 'Y'))
for row in retrieved_catalog:
    print('{:8.2f} {:8.2f}'.format(row['x'], row['y']))

In [ ]:
# Remove catalog using the API
w.remove_catalog(catalog_label='sample')
print("Removed sample catalog")

### Programmatic catalog loading and styling using the astro-image-display-api.

In [ ]:
# Create sample catalogs with different styles
# First 2 points as red circles, rest as cyan crosses
red_catalog = Table([x_coords[:2], y_coords[:2]], names=['x', 'y'])
w.load_catalog(red_catalog, catalog_label='red_circles')
w.set_catalog_style(catalog_label='red_circles', color='red', shape='circle', size=50, linewidth=2)

# Rest as cyan crosses
cyan_catalog = Table([x_coords[2:], y_coords[2:]], names=['x', 'y'])
w.load_catalog(cyan_catalog, catalog_label='cyan_crosses')
w.set_catalog_style(catalog_label='cyan_crosses', color='cyan', shape='cross', size=20)

In [ ]:
# Clear existing catalogs and load new one with SkyCoord
try:
    w.remove_catalog(catalog_label='red_circles')
except ValueError:
    pass  # Catalog doesn't exist
try:
    w.remove_catalog(catalog_label='cyan_crosses')
except ValueError:
    pass  # Catalog doesn't exist

# Create catalog with SkyCoord coordinates  
from astropy.coordinates import SkyCoord
from astropy import units as u

# Convert x,y to approximate RA/Dec for demonstration
ra_vals = (x_coords / 100 + 19.5) * u.deg  # Approximate RA
dec_vals = (y_coords / 100 + 0.2) * u.deg  # Approximate Dec
skycoord_table = Table([SkyCoord(ra_vals, dec_vals, frame=frame)], names=['coord'])

w.load_catalog(skycoord_table, catalog_label='skycoord_markers', use_skycoord=True)

In [ ]:
# Clear all catalogs using the API
try:
    w.remove_catalog(catalog_label='skycoord_markers')
    print("Cleared all markers using catalog API")
except ValueError:
    print("No skycoord_markers catalog to remove")

### Using widgets to drive image display

In [ ]:
# Generate random "stars" using the image from the API
import numpy as np
from astropy.table import Table

# Maximum number of "stars" to generate randomly
max_stars = 1000

# Number of pixels from edge to avoid
dpix = 20

# Get image data using API
img_data = w.get_image()
if hasattr(img_data, 'shape'):
    shape = img_data.shape
elif hasattr(img_data, 'data') and hasattr(img_data.data, 'shape'):
    shape = img_data.data.shape
else:
    # Fallback
    shape = (256, 256)
    
# Random "stars" generated  
bad_locs = np.random.randint(
    dpix, high=shape[1] - dpix, size=[max_stars, 2])

# Only want those not near the edges
mask = ((dpix < bad_locs[:, 0]) &
        (bad_locs[:, 0] < shape[0] - dpix) &
        (dpix < bad_locs[:, 1]) &
        (bad_locs[:, 1] < shape[1] - dpix))
locs = bad_locs[mask]

# Put them in table
t = Table([locs[:, 1], locs[:, 0]], names=('x', 'y'))
print(f"Generated {len(t)} random star positions")


In [ ]:
# Load the "stars" as a catalog
w.load_catalog(t, catalog_label='stars')

The following illustrates how to control number of markers displayed using interactive widget from `ipywidgets`.

In [ ]:
import ipywidgets as ipyw

# Set catalog style and create function to control display
w.set_catalog_style(catalog_label='stars', color='red', shape='circle', size=10, linewidth=2)
output = ipyw.Output()
# Show the slider widget.
# Define a function to control marker display using catalog API
def show_circles(n):
    """Show subset of stars using catalog API."""
    # Remove existing catalog
    try:
        w.remove_catalog(catalog_label='stars')
    except ValueError as exc:
        raise exc  # Catalog doesn't exist
        
    # Create subset catalog
    t2show = t[:n]
    w.load_catalog(t2show, catalog_label='stars')
    w.set_catalog_style(catalog_label='stars', color='red', shape='circle', size=10, linewidth=2)
    
    with output:
        print('Displaying {} markers...'.format(len(t2show)))

We redisplay the image widget below above the slider. Note that the slider affects both this view of the image widget and the one near the top of the notebook.

In [ ]:
from IPython.display import display

import ipywidgets as ipyw
from ipywidgets import interactive


# Show the slider widget.
slider = interactive(show_circles,
                     n=ipyw.IntSlider(min=0,max=len(t),step=1,value=0, continuous_update=False))
display(w, slider, output)

Now, use the slider. The chosen `n` represents the first `n` "stars" being displayed.

## Interactive features like click to center are astrowidgets-specific and not part of the astro-image-display-api

The following cell changes the visibility or position of the cursor info bar.


In [ ]:
w.cursor = 'top'  # 'top', 'bottom', None
print(w.cursor)

In [ ]:
# Note: click_center is not part of the astro-image-display-api
# This feature is bqplot-specific and not available through the API
print("Click to center is a bqplot-specific feature not in the API")

In [ ]:
# Interactive marking is bqplot-specific
print("Interactive marking features are bqplot-specific")

In [ ]:
# Note: start_marking/is_marking is not part of the astro-image-display-api
# Interactive marking is bqplot-specific
print("Interactive marking is a bqplot-specific feature not in the API")

In [ ]:
# Interactive marking is not part of the astro-image-display-api
# The API uses catalog-based marking instead
print("Interactive marking is a bqplot-specific feature not in the API")
print("Use catalog-based marking instead with load_catalog()")